In [1]:
# Importing pandas for data loading and manipulation
import pandas as pd

In [2]:
# Loading the raw Titanic training dataset
data = pd.read_csv("../datasets/raw/titanic/train.csv")

data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Separating Features and Target

In supervised machine learning, the dataset is separated into:

- **Features (`X`)**: Input variables used by the model to learn patterns.
- **Target (`y`)**: The output variable that the model tries to predict.

For the Titanic dataset, `Survived` is the target variable:

- `0` = Did not survive
- `1` = Survived

All remaining columns initially form the feature set `X`. Further preprocessing will be applied before model training.

In [3]:
# Separating the input features from the target variable
X = data.drop(columns=['Survived'])
y = data['Survived']

In [4]:
# Cheeckking the saparated features and target values
display(X.head())
print("Target values:")
display(y.head())

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Target values:


0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [5]:
#checking the dimensions of the features and target values
print("X shape", X.shape)
print("y shape", y.shape)

X shape (891, 11)
y shape (891,)


In [6]:
# Comparing the dimensions and data types of X and y
print("X type:", type(X))
print("X dimensions:", X.ndim)
print("X shape:", X.shape)

print()

print("y type:", type(y))
print("y dimensions:", y.ndim)
print("y shape:", y.shape)

X type: <class 'pandas.DataFrame'>
X dimensions: 2
X shape: (891, 11)

y type: <class 'pandas.Series'>
y dimensions: 1
y shape: (891,)


### Understanding the Structure of `X` and `y`

`X` is a 2D DataFrame containing the input features, while `y` is a 1D Series containing one target value for each passenger.

In Pandas:

- `data['Survived']` returns a **Series** with shape `(891,)`.
- `data[['Survived']]` returns a **DataFrame** with shape `(891, 1)`.

The values may be the same, but their data structures and dimensions are different.

In [7]:
# Comparing single-bracket and double-bracket column selection
y_series = data['Survived']
y_dataframe = data[['Survived']]

print("Single bracket:")
print("Type:", type(y_series))
print("Shape:", y_series.shape)
print("Dimensions:", y_series.ndim)

print()

print("Double bracket:")
print("Type:", type(y_dataframe))
print("Shape:", y_dataframe.shape)
print("Dimensions:", y_dataframe.ndim)

Single bracket:
Type: <class 'pandas.Series'>
Shape: (891,)
Dimensions: 1

Double bracket:
Type: <class 'pandas.DataFrame'>
Shape: (891, 1)
Dimensions: 2


## Train-Test Split

A machine learning model should be evaluated on data that it did not see during training.

The dataset is therefore divided into:

- **Training set:** Used by the model to learn patterns.
- **Test set:** Used to evaluate how well the trained model performs on unseen data.

Evaluating a model on the same data used for training can give an overly optimistic result.

**Generalization** refers to a model's ability to perform well on new, unseen data.

A model may **overfit** when it performs very well on the training data but poorly on unseen data.

In [8]:
# Checking the class distribution of the target variable
print(y.value_counts())

print()

print(y.value_counts(normalize=True))

Survived
0    549
1    342
Name: count, dtype: int64

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [9]:
from sklearn.model_selection import train_test_split

# Splitting the features and target into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [10]:
# Checking the sizes of the training and test sets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (712, 11)
X_test shape: (179, 11)
y_train shape: (712,)
y_test shape: (179,)


In [11]:
# Comparing the class  proportions across the original, training, and test sets
class_distribution = pd.DataFrame({
    'Original' : y.value_counts(normalize=True),
    'Training' : y_train.value_counts(normalize=True),
    'Test' : y_test.value_counts(normalize=True)
})*100

class_distribution.round(2)

,Original,Training,Test
Survived,,,
0,61.62,61.66,61.45
1,38.38,38.34,38.55


In [12]:
# Checking that the training and test sets contain different rows
print("Overlapping rows:", len(set(X_train.index) & set(X_test.index)))

Overlapping rows: 0


### Understanding `random_state`

`train_test_split()` randomly shuffles the data before creating the training and test sets.

`random_state` fixes the random seed so that the same split can be reproduced when the code is run again with the same data and parameters.

The number `42` has no special machine learning meaning; any fixed integer can be used.

In [13]:
# Comparing two splits created with the same random state
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Same training rows:", X_train_1.index.equals(X_train_2.index))
print("Same test rows:", X_test_1.index.equals(X_test_2.index))

Same training rows: True
Same test rows: True


In [14]:
# Comparing splits created with different random states
X_train_42, X_test_42, _, _ = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train_100, X_test_100, _, _ = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=100,
    stratify=y
)

print("Same training rows:", X_train_42.index.equals(X_train_100.index))
print("Same test rows:", X_test_42.index.equals(X_test_100.index))

Same training rows: False
Same test rows: False


`random_state` controls the reproducibility of the random shuffle, while `stratify=y` helps preserve the target class proportions in both the training and test sets.

By default, `train_test_split()` uses `shuffle=True`.